<a href="https://colab.research.google.com/github/mrdantzig/nlp-bible-code/blob/master/22%EC%9E%A5_%EC%8B%AC%EC%B8%B5%ED%95%99%EC%8A%B5%EC%9D%84_%EC%9D%B4%EC%9A%A9%ED%95%9C_%EC%9D%98%EB%AF%B8%EB%B6%84%EC%84%9D/%5B22-2%5D%20%EC%8B%AC%EC%B8%B5%ED%95%99%EC%8A%B5%20%EA%B8%B0%EB%B0%98%20%EC%9D%98%EB%AF%B8%EC%97%AD%28Semantic%20Role%20Labeling%2CSRL%29%20%EC%8B%A4%EC%8A%B5_%EC%88%98%EC%A0%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [22-2] 심층학습 기반 의미역(Semantic Role Labeling, SRL) 실습 (수정본)

자연어처리 바이블 22장 실습 [22-2]를 최신 Colab에서 실행되도록 고친 노트북입니다. BERT 기반 의미역 결정(SRL) 사전학습 모델로 문장의 **술어(동사)별 논항(ARG0·ARG1·ARGM-…)**을 BIO 태그로 찾습니다(교재 22.1).

**원본 대비 수정 사항**
1. **Python 3.9 별도 가상환경에서 실행**: AllenNLP는 2022년에 개발이 끝나 최신 Colab(Python 3.12)에는 설치되지 않습니다. `uv`로 Python 3.9 가상환경을 만들어 `allennlp==2.10.1`, `allennlp-models==2.10.1`(마지막 버전)을 설치하고, 의미역 결정은 그 환경에서 스크립트로 실행합니다. 결과는 JSON으로 받아 이 노트북에서 확인합니다.
2. **모델 주소 변경**: 원본의 `s3-us-west-2.amazonaws.com/allennlp/models/bert-base-srl-2019.06.17.tar.gz`(allennlp 0.9용)는 2.x에서 읽을 수 없어, AllenNLP 공식 공개 모델 `structured-prediction-srl-bert`(2020.12.15)로 바꿨습니다.
3. **`pip install sklearn` 삭제**: 설치 오류가 나는 패키지 이름이며 이 실습에는 필요 없습니다.
4. **결과 보기 추가**: 원본의 `{토큰 위치: 태그}` 딕셔너리 출력은 그대로 두고, 술어별 의미역을 표와 문장 형태로 함께 보여 줍니다.

※ 처음 실행 시 가상환경 설치(3~5분)와 모델 다운로드(약 400MB)가 필요합니다. GPU는 필요 없습니다.

## 1. Python 3.9 가상환경과 AllenNLP 설치

In [ ]:
!pip install -q uv
!uv venv /content/srl-env --python 3.9 -q
# allennlp 2.10.1과 함께 동작하는 버전으로 고정
#  - typing_extensions<4.6: spaCy 3.3이 쓰는 pydantic 1.8과 충돌 방지
#  - en_core_web_sm: SRL predictor의 단어 분리에 필요한 spaCy 영어 모델
!uv pip install -q --python /content/srl-env/bin/python \
    "allennlp==2.10.1" "allennlp-models==2.10.1" "huggingface_hub==0.10.1" "numpy<1.24" \
    "typing_extensions<4.6" "setuptools<70" \
    "en_core_web_sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.3.0/en_core_web_sm-3.3.0-py3-none-any.whl"
!/content/srl-env/bin/python -c "import allennlp, allennlp_models; print('allennlp', allennlp.__version__)"

## 2. 의미역 결정 스크립트

원본의 `srl_tag_result` 함수를 그대로 사용합니다. 가상환경의 Python으로 실행하기 위해 파일로 저장합니다.

In [ ]:
%%writefile srl_run.py
import json
import sys

from allennlp.predictors.predictor import Predictor
import allennlp_models.structured_prediction  # noqa: F401  (SRL predictor 등록)

# 수정: allennlp 0.9용 2019년 모델 → AllenNLP 2.x 공식 BERT SRL 모델
MODEL_URL = ("https://storage.googleapis.com/allennlp-public-models/"
             "structured-prediction-srl-bert.2020.12.15.tar.gz")
predictor_srl = Predictor.from_path(MODEL_URL)


# srl 결과 dict로 반환 (원본과 동일)
def srl_tag_result(predictor_srl, sen):
    dict_result = predictor_srl.predict(sentence=sen)
    result_dict_srl = dict()
    for idx_tag, tags in enumerate(dict_result["verbs"]):
        vals_dict = dict()
        alone_vnode = False
        start_node = 0
        str_temp_val = str()
        for idx_val, val in enumerate(tags["tags"]):
            if "-V" in val:
                start_node = idx_val
                str_temp_val = val
                alone_vnode = True
            else:
                if ("B-" in val) or ("I-" in val):
                    vals_dict[idx_val + 1] = val
                    alone_vnode = False
        if alone_vnode is False:
            try:
                result_dict_srl[start_node + 1].update(vals_dict)
            except KeyError:
                result_dict_srl[start_node + 1] = vals_dict
    return result_dict_srl


if __name__ == "__main__":
    sentences = json.load(open(sys.argv[1], encoding="utf-8"))
    outputs = []
    for sen in sentences:
        pred = predictor_srl.predict(sentence=sen)
        outputs.append({"sentence": sen,
                        "words": pred["words"],
                        "verbs": [{"verb": v["verb"], "description": v["description"], "tags": v["tags"]}
                                  for v in pred["verbs"]],
                        "result_dict": srl_tag_result(predictor_srl, sen)})
    json.dump(outputs, open(sys.argv[2], "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print("done:", len(outputs), "sentences")

## 3. Main — 문장 입력과 실행

In [ ]:
import json

sentences = [
    "More than a few CEOs say the red-carpet treatment tempts them to return to a heartland city for future meetings.",   # 원본 예문
    "The gardener planted some trees in my yard yesterday.",                                                          # (추가) 예문
]
json.dump(sentences, open("srl_input.json", "w", encoding="utf-8"))

!/content/srl-env/bin/python srl_run.py srl_input.json srl_output.json

results = json.load(open("srl_output.json", encoding="utf-8"))
# 원본과 같은 형식의 출력: {술어 위치: {토큰 위치: BIO 태그}}  (위치는 1부터)
# (JSON을 거치며 문자열이 된 키를 원본처럼 정수로 되돌림)
result_dict_srl = {int(k): {int(i): t for i, t in v.items()} for k, v in results[0]["result_dict"].items()}
print(result_dict_srl)

## 4. (추가) 결과 읽기

- **ARG0**: 행위자(…가), **ARG1**: 대상(…을), **ARG2~4**: 도착점·수혜자 등, **ARGM-TMP/LOC/PRP**: 시간·장소·목적 부가어
- 원본 출력의 `6: {1: 'B-ARG0', …}`는 "6번째 단어(say)를 술어로 볼 때 1번째 단어부터 ARG0가 시작된다"는 뜻입니다.

In [ ]:
import pandas as pd

for r in results:
    print("문장:", r["sentence"])
    for v in r["verbs"]:
        print("  -", v["description"])
    table = pd.DataFrame({"word": r["words"]})
    for v in r["verbs"]:
        table[v["verb"]] = v["tags"]
    table.index = table.index + 1          # 원본처럼 위치를 1부터 표시
    display(table)